# Demo 2 — Tre segnali controllati, non tre strumenti reali

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
rng = np.random.default_rng(0)
t = np.arange(FS)/FS
segnali = {
    "Tono 330 Hz": al.sine(330, 1, FS, amplitude=1.0),
    "Somma armonica": al.sum_harmonics(range(1,10), f0=330, duration=1, sr=FS),
    "Rumore": al.white_noise(len(t),seed=0),
}
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ax, (nome, y) in zip(axes, segnali.items()):
    y = y/al.rms(y)*0.05
    segnali[nome] = y
    A = np.abs(al.analysis.stft(y, n_fft=1024, hop_length=512))
    P = A**2
    centroide = al.analysis.feature.spectral_centroid(S=A, sr=FS).mean()
    rolloff = al.analysis.feature.spectral_rolloff(S=P, sr=FS, roll_percent=0.85).mean()
    flat = al.analysis.feature.spectral_flatness(S=P, power=1.0).mean()
    zcr = al.analysis.feature.zero_crossing_rate(y).mean()
    print(f"{nome}: RMS 0.050; ZCR {zcr:.3f}; centroide {centroide:.0f} Hz; "
          f"rolloff {rolloff:.0f} Hz; flatness {flat:.3f}")
    ax.plot(al.analysis.fft_frequencies(sr=FS, n_fft=1024), A.mean(axis=1))
    ax.set_ylabel(nome); ax.set_xlim(0, FS/2)
axes[-1].set_xlabel("Frequenza [Hz]")
fig.suptitle("Spettro medio in ampiezza: stessa scala di frequenza")
plt.tight_layout(); plt.show()
